# VIX pull

Fetches the CBOE volatility indices from FRED and writes them to `data/raw/`. Implied
volatility is the best-documented predictor of realised volatility and consistently
beats backward-looking estimators, and nothing else in the feature set is
forward-looking, so it is the one input here that carries information the price history
cannot.

Two series:

- `VIXCLS`, 30-day implied volatility on the S&P 500
- `VXVCLS`, the 3-month equivalent, whose ratio to VIX gives a term-structure slope

**Publication timing matters.** These are daily closes stamped at 16:15 ET. A bar can
only use a close that has already been published, so the merge in
`feature_engineering.py` is done against a publication timestamp rather than a date.

Run this once, or whenever the sample is extended. It writes raw data and nothing else.

In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from fredapi import Fred

RAW = Path("../data/raw")
SERIES = {"vix": "VIXCLS", "vix3m": "VXVCLS"}
PUBLISH_ET = pd.Timedelta(hours=16, minutes=15)     # CBOE close stamp

load_dotenv(Path("../.env"))
fred = Fred(api_key=os.environ["FRED_API_KEY"])

In [2]:
frames = {}
for name, code in SERIES.items():
    s = fred.get_series(code).rename(name)
    s.index = pd.to_datetime(s.index)
    frames[name] = s
    print(f"{code:<8} {len(s):>6} observations  {s.index.min().date()} -> {s.index.max().date()}")

vix = pd.concat(frames.values(), axis=1).sort_index()
vix.index.name = "date"

VIXCLS     9576 observations  1990-01-02 -> 2026-09-15


VXVCLS     4901 observations  2007-12-04 -> 2026-09-15


Holidays and the odd missing print leave gaps. Forward-filling would invent a close
that was never published, so rows without a VIX level are dropped instead and the
as-of merge downstream carries the last real value.

In [3]:
before = len(vix)
vix = vix.dropna(subset=["vix"])
print(f"dropped {before - len(vix)} rows with no VIX close")
print(f"rows with no 3-month value: {vix.vix3m.isna().sum()} "
      f"(series starts {frames['vix3m'].index.min().date()})")

# publication instant, in UTC, so downstream merges cannot peek
vix["published_utc"] = (vix.index + PUBLISH_ET).tz_localize("America/New_York").tz_convert("UTC")

dropped 302 rows with no VIX close
rows with no 3-month value: 4550 (series starts 2007-12-04)


In [4]:
out = RAW / "vix_daily.parquet"
vix.to_parquet(out)
print(f"wrote {out}  {len(vix):,} rows x {len(vix.columns)} columns")
print(vix.tail(3).to_string())
print()
print(vix[["vix", "vix3m"]].describe().round(2).to_string())

wrote ../data/raw/vix_daily.parquet  9,274 rows x 3 columns
              vix  vix3m             published_utc
date                                              
2026-09-11  15.84  18.60 2026-09-11 20:15:00+00:00
2026-09-14  17.10  19.28 2026-09-14 20:15:00+00:00
2026-09-15  17.20  19.36 2026-09-15 20:15:00+00:00

           vix    vix3m
count  9274.00  4724.00
mean     19.43    21.60
std       7.72     7.59
min       9.14    11.85
25%      13.99    16.27
50%      17.58    19.84
75%      22.68    24.65
max      82.69    72.98


**Findings:** 9,576 daily VIX closes back to 1990 and 4,901 three-month closes back to
December 2007, so both cover the 2016 onward modelling sample in full. VIX averages
19.5 over the whole history with a maximum of 82.7 in March 2020. The file is raw data
and gitignored, so anyone rebuilding the pipeline runs this notebook once.